In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(name)s — %(levelname)s — %(message)s',
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import subprocess
import sys
import time as _time
from pathlib import Path
from scipy.spatial.distance import cdist
import xarray as xr

from pyS3M import PlottingBase

plotter = PlottingBase.PublicationPlotter(dark_background=False)

from pyS3M.simulation.multicolour import sample_chip_patches_batch, sanitise_calibration_maps

# Shared with the per-task worker so notebook and worker can't drift apart
from _bayer_noise_sweep_task import (
    CHIP_SAMPLING_SEED,
    DEFAULT_CALIB_DIR,
    STARTING_FLAG,
    load_calibration,
    photon_space,
)

FAST_MODE = True  # False -> the published 200-photon-level x 100,000-bootstrap sweep
print(f"FAST_MODE={FAST_MODE}")

# ── Background level ──────────────────────────────────────────────────────────
# Mean background photons sensed by each pixel per frame (any colour, any dye)
BACKGROUND_PHOTONS = 5.0
print(f"Background: {BACKGROUND_PHOTONS:g} photons/pixel/frame")

# Raw results (HDF5, ~2.4 GB per dye in full mode) go to the lab backup drive
RAW_ROOT = Path("/media/u2473660/JSBLeeLabBackup/Data/20261002_BayesianComparison")
FIGURE_DIR = Path("figures")  # git-ignored (*.svg)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Background goes into the folder name so runs at different backgrounds never overwrite
sweep_tag = f"{'fast' if FAST_MODE else 'full'}_bg{BACKGROUND_PHOTONS:g}".replace(".", "p")
save_folder = RAW_ROOT / sweep_tag
save_folder.mkdir(parents=True, exist_ok=True)
print(f"Raw results will be saved to: {save_folder}")

# FigureSI_ — Bayer precision with per-bootstrap real-chip camera noise

Precision of the standard Bayer (RGGB) pattern when every bootstrap frame sees **real Ximea
sensor noise**: each frame gets its own randomly located 14×14 crop of the full-chip
gain/offset/variance/rqe maps, so hot or defective pixels appear at their true frequency.

Each bootstrap sample's crop is reused at every photon level (like its ground-truth position).
The same crop generates the frame and is used to convert and weight it for fitting, including
per-pixel read noise $\sqrt{\mathrm{variance}}/\mathrm{gain}$, as the real pipeline would. Crop
offsets are saved per dye (`*_chipcrop_offsets.csv`). The 3 physically invalid chip pixels
(gain or rqe ≤ 0) are set to the chip median; hot pixels are kept.

**Background (`BACKGROUND_PHOTONS`).** The mean number of background photons each pixel senses
per frame (Poisson, detected with QE = 1), the same for every pixel colour and every dye.
This definition is newer than the published SI sweeps, where `background_photons = 5` meant
5/3 photons per pixel. Set `BACKGROUND_PHOTONS = 5/3` to match them.

**Sweep.** Dyes ATTO 488, ATTO 565 and ATTO 647N; 200 photon levels from 200 to 50,000,
log-spaced; 100,000 bootstraps per level. Each dye runs in its own subprocess
(`_bayer_noise_sweep_task.py`). `FAST_MODE = True` runs 10 levels × 2,000 bootstraps
(minutes); `False` runs the full sweep (hours). Raw results go to
`RAW_ROOT/{fast,full}_bg<BACKGROUND_PHOTONS>/`.


In [ ]:
# ── Real Ximea calibration: how often are crops affected? ─────────────────────
calib = load_calibration(DEFAULT_CALIB_DIR)
chip_maps, n_invalid = sanitise_calibration_maps(
    {k: calib[k] for k in ("gain", "offset", "variance", "rqe")}
)
image_size = 14  # Bayer 2x2 unit cell x n_unit_cells=7, as in the sweep

HOT_VARIANCE = 10.0  # ADU^2; ~10x the chip median -- what we call a "hot" pixel below
var_med = float(np.median(chip_maps["variance"]))
hot_frac = float(np.mean(chip_maps["variance"] > HOT_VARIANCE))
print(f"chip shape {chip_maps['gain'].shape}, invalid pixels replaced: {n_invalid}")
print(f"median variance {var_med:.3f} ADU^2, max {chip_maps['variance'].max():.0f} ADU^2")
print(f"pixels with variance > {HOT_VARIANCE:g}: {hot_frac:.2e} of chip")
print(f"-> P(a {image_size}x{image_size} crop contains one) ≈ "
      f"{1 - (1 - hot_frac) ** (image_size ** 2):.3f}  (independent-pixel estimate)")

In [ ]:
# ── Example crops: variance maps for a few bootstrap frames ───────────────────
# Uses the worker's seed, so these are frames 0..n-1 of the real sweep's crop draw.
n_show = 6
crops, r0, c0 = sample_chip_patches_batch(
    chip_maps, (image_size, image_size), 2000, np.random.default_rng(CHIP_SAMPLING_SEED)
)
crop_max_var = crops["variance"].reshape(len(r0), -1).max(axis=1)
hot_idx = np.flatnonzero(crop_max_var > HOT_VARIANCE)
show = list(np.flatnonzero(crop_max_var <= HOT_VARIANCE)[: n_show - 2]) + list(hot_idx[:2])

fig, axs = plt.subplots(1, n_show, figsize=(6.69, 1.4))
vmax = np.log10(max(crop_max_var[show]))
for ax, k in zip(axs, show):
    im = ax.imshow(np.log10(crops["variance"][k]), cmap="magma",
                   vmin=np.log10(var_med) - 0.3, vmax=vmax, origin="lower")
    ax.set_title(f"frame {k}\n({r0[k]}, {c0[k]})", fontsize=7)
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axs, fraction=0.02, label=r"log$_{10}$ variance / ADU$^2$")
plt.savefig(FIGURE_DIR / "bayer_noise_example_crops.svg", dpi=300, format="svg")
plt.show()
print(f"{len(hot_idx)}/{len(r0)} of these crops contain a pixel with variance > {HOT_VARIANCE:g}")

In [ ]:
# ── Sweep configuration ───────────────────────────────────────────────────────
sim_pixel_size = 69
sim_dyes = ["ATTO 488", "ATTO 565", "ATTO 647N"]

N_PHOTON_STEPS = 10 if FAST_MODE else 200
N_BOOTSTRAP = 2_000 if FAST_MODE else 100_000

n_photon_space = photon_space(N_PHOTON_STEPS)  # 200 -> 50,000, log-spaced
print(f"Dyes: {sim_dyes}")
print(f"Photon levels: {len(n_photon_space)}  ({n_photon_space[0]:.0f}–{n_photon_space[-1]:.0f})")
print(f"Bootstraps per level: {N_BOOTSTRAP:,}")
print(f"Background: {BACKGROUND_PHOTONS:g} photons/pixel/frame")


In [ ]:
# ── Run simulations: one subprocess per dye ──────────────────────────────────
sweep_script = os.path.join(os.getcwd(), "_bayer_noise_sweep_task.py")
t0 = _time.time()

for i_task, dye in enumerate(sim_dyes, start=1):
    print(f"[{i_task}/{len(sim_dyes)}]  {dye}", end="  ", flush=True)
    result = subprocess.run(
        [
            sys.executable, sweep_script,
            "--dye", dye,
            "--save-folder", str(save_folder),
            "--n-bootstrap", str(N_BOOTSTRAP),
            "--n-photon-levels", str(N_PHOTON_STEPS),
            "--background-photons", repr(BACKGROUND_PHOTONS),
        ],
        capture_output=True, text=True,
    )
    elapsed = (_time.time() - t0) / 60
    eta = (len(sim_dyes) - i_task) * elapsed / i_task
    if result.returncode == 0:
        print(f"done  ({elapsed:.1f} min, ETA {eta:.1f} min)")
    else:
        print(f"FAILED  (exit {result.returncode}, {elapsed:.1f} min)")
        print(result.stderr[-2000:])

print(f"\nAll simulations complete in {(_time.time() - t0) / 60:.1f} min.")


In [ ]:
# ── Load results → Chip_Noise_XR ──────────────────────────────────────────────
# Dims: [dye, n_photons, metric]. sigma_xy is also split by whether that bootstrap's
# crop contains a hot pixel.
_metrics = ["sigma_xy", "sigma_colour", "sigma_xy_hotcrop", "sigma_xy_cleancrop"]
Chip_Noise_XR = xr.DataArray(
    np.full([len(sim_dyes), len(n_photon_space), len(_metrics)], np.nan),
    coords={"dye": sim_dyes, "n_photons": n_photon_space, "metric": _metrics},
    dims=["dye", "n_photons", "metric"],
)
hot_fraction = {}

def _sigma_xy(ex, ey):
    return np.sqrt((np.nanstd(ex) ** 2 + np.nanstd(ey) ** 2) / 2) * sim_pixel_size

for i_d, dye in enumerate(sim_dyes):
    stem = save_folder / f"{STARTING_FLAG}LM_method_{dye.replace('/', '-')}_"
    raw_file = Path(f"{stem}rawresults.h5")
    gt_file = Path(f"{stem}fittesting_input_groundtruthpositions.csv")
    ip_file = Path(f"{stem}fittesting_input_parameters.csv")
    off_file = Path(f"{stem}fittesting_chipcrop_offsets.csv")
    if not all(f.exists() for f in (raw_file, gt_file, ip_file, off_file)):
        print(f"Missing files for {dye}")
        continue

    gt = pd.read_csv(gt_file)
    gt_px = np.vstack([gt["x0"].to_numpy(), gt["y0"].to_numpy()]).T / sim_pixel_size
    dye_BGR = pd.read_csv(ip_file).to_numpy()[0][-3:]
    colour_true = np.expand_dims(dye_BGR / dye_BGR.sum(), 0)

    off = pd.read_csv(off_file)
    crops_v, _, _ = sample_chip_patches_batch(
        {"variance": chip_maps["variance"]}, (image_size, image_size), len(off), None,
        offsets=(off["row0"].to_numpy(), off["col0"].to_numpy()),
    )
    hot_crop = crops_v["variance"].reshape(len(off), -1).max(axis=1) > HOT_VARIANCE
    hot_fraction[dye] = hot_crop.mean()

    df_raw = pd.read_hdf(raw_file)
    for k in range(len(n_photon_space)):
        sub = df_raw[df_raw["photon_level"] == k]
        frame = sub["frame"].to_numpy().astype(int)
        filt = ~(
            (sub["xc"] < 0) | (sub["yc"] < 0) | (sub["s_x"] < 0) | (sub["s_y"] < 0) |
            (sub["xc"] > image_size) | (sub["yc"] > image_size)
        ).to_numpy()
        if filt.sum() < 5:
            continue
        ex = sub["xc"].to_numpy() - gt_px[frame, 0]
        ey = sub["yc"].to_numpy() - gt_px[frame, 1]
        Chip_Noise_XR[i_d, k, 0] = _sigma_xy(ex[filt], ey[filt])
        colour = sub[["A_B", "A_G", "A_R"]].to_numpy()[filt]
        Chip_Noise_XR[i_d, k, 1] = float(np.nanstd(cdist(colour, colour_true)))
        for j, sel in ((2, hot_crop[frame]), (3, ~hot_crop[frame])):
            if (filt & sel).sum() >= 5:
                Chip_Noise_XR[i_d, k, j] = _sigma_xy(ex[filt & sel], ey[filt & sel])

Chip_Noise_XR.to_netcdf(save_folder / "Chip_Noise_XR.nc")
print(f"Saved  shape={Chip_Noise_XR.shape}  dims={list(Chip_Noise_XR.dims)}")
print({d: f"{f:.3f}" for d, f in hot_fraction.items()}, "<- fraction of crops with a hot pixel")


In [ ]:
# ── σ_xy and σ_colour vs N_photons ────────────────────────────────────────────
Chip_Noise_XR = xr.load_dataarray(save_folder / "Chip_Noise_XR.nc")

fs = 8
fig, axs = plotter.two_column_plot(ncols=3, nrows=2, width_ratios=[1, 1, 1], height_ratios=[1, 1])

for d_num, dye in enumerate(sim_dyes):
    sel = lambda metric: Chip_Noise_XR.sel(dye=dye, metric=metric)
    ax = axs[0, d_num]
    ax = plotter.line_plot(ax, n_photon_space, sel("sigma_xy"),
                           color="black", linewidth=1.5, label="all crops")
    ax = plotter.line_plot(ax, n_photon_space, sel("sigma_xy_cleancrop"),
                           color="#4363d8", linewidth=1.0, linestyle="--", label="no hot pixel")
    ax = plotter.line_plot(ax, n_photon_space, sel("sigma_xy_hotcrop"),
                           color="#f58231", linewidth=1.0, linestyle="--", label="hot pixel in crop")
    ax.set_ylabel(r"$\sigma_{xy}$ / nm", fontsize=fs)
    ax.set_title(dye, fontsize=fs)
    ax.legend(loc="best", fontsize=fs - 2, framealpha=0.7)

    ax = axs[1, d_num]
    ax = plotter.line_plot(ax, n_photon_space, sel("sigma_colour"), color="black", linewidth=1.5)
    ax.set_ylabel(r"$\sigma_{\rm colour}$", fontsize=fs)

for ax in axs.ravel():
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel(r"$N_{\rm photons}$", fontsize=fs)
    ax.set_xlim([n_photon_space[0], n_photon_space[-1]])
    ax.tick_params(labelsize=fs - 1)

plt.savefig(FIGURE_DIR / "bayer_chip_noise_precision.svg", dpi=300, format="svg")
plt.show()
